In [2]:
!git clone https://github.com/Mahim021/NLP.git /kaggle/working/mailsense

Cloning into '/kaggle/working/mailsense'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 125 (delta 25), reused 123 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 520.11 KiB | 21.67 MiB/s, done.
Resolving deltas: 100% (25/25), done.


In [3]:
%cd /kaggle/working/mailsense
!git status

/kaggle/working/mailsense
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [4]:
from pathlib import Path

ROOT = Path("/kaggle/working/mailsense")

for path in [
    ROOT / "src",
    ROOT / "configs",
    ROOT / "data" / "Ask0729-fixed.txt",
    ROOT / "requirements.txt",
]:
    print(path, "->", path.exists())

/kaggle/working/mailsense/src -> True
/kaggle/working/mailsense/configs -> True
/kaggle/working/mailsense/data/Ask0729-fixed.txt -> True
/kaggle/working/mailsense/requirements.txt -> True


In [5]:
%cd /kaggle/working/mailsense

!pip install -q -r requirements.txt

/kaggle/working/mailsense


In [6]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [7]:
from pathlib import Path

ROOT = Path("/kaggle/working/mailsense")

print("Repository:", ROOT)
print("Dataset:", ROOT / "data" / "Ask0729-fixed.txt")
print("LSTM config:", ROOT / "configs" / "lstm.json")
print("Word2Vec:", ROOT / "embeddings" / "word2vec.bin")

print("\nWord2Vec exists:",
      (ROOT / "embeddings" / "word2vec.bin").exists())

Repository: /kaggle/working/mailsense
Dataset: /kaggle/working/mailsense/data/Ask0729-fixed.txt
LSTM config: /kaggle/working/mailsense/configs/lstm.json
Word2Vec: /kaggle/working/mailsense/embeddings/word2vec.bin

Word2Vec exists: False


In [18]:
!wget -q --show-progress \
  "https://huggingface.co/LoganKilpatrick/GoogleNews-vectors-negative300/resolve/main/GoogleNews-vectors-negative300.bin.gz?download=true" \
  -O /kaggle/working/GoogleNews-vectors-negative300.bin.gz

/kaggle/working/Goo 100%[===================>]   1.53G   109MB/s    in 19s     


In [19]:
from pathlib import Path

p = Path("/kaggle/working/GoogleNews-vectors-negative300.bin.gz")

print("Exists:", p.exists())
print("Bytes:", p.stat().st_size)
print("GB:", round(p.stat().st_size / 1024**3, 2))

Exists: True
Bytes: 1647046227
GB: 1.53


In [20]:
!file /kaggle/working/GoogleNews-vectors-negative300.bin.gz

/kaggle/working/GoogleNews-vectors-negative300.bin.gz: gzip compressed data, was "GoogleNews-vectors-negative300.bin", last modified: Thu Dec 12 00:12:48 2013, max compression, from Unix, original size modulo 2^32 3644258522


In [21]:
%cd /kaggle/working/mailsense
!python -m src.preprocessing.prepare_data

/kaggle/working/mailsense
{
  "dataset_file": "/kaggle/working/mailsense/data/Ask0729-fixed.txt",
  "raw_rows": 3657,
  "malformed_or_invalid_rows": 0,
  "raw_class_distribution": {
    "No": 1938,
    "Yes": 1719
  },
  "seed": 42,
  "final_rows": 3651,
  "final_class_distribution": {
    "No": 1933,
    "Yes": 1718
  },
  "train_size": 2555,
  "train_class_distribution": {
    "No": 1353,
    "Yes": 1202
  },
  "val_size": 548,
  "val_class_distribution": {
    "No": 290,
    "Yes": 258
  },
  "test_size": 548,
  "test_class_distribution": {
    "No": 290,
    "Yes": 258
  },
  "removed_total": 6
}


In [22]:
import pandas as pd
from pathlib import Path
from src.lstm.train_lstm import build_vocab

train = pd.read_csv("data/splits/train.csv")

vocab = build_vocab(
    train["text"].tolist(),
    min_freq=2,
    max_size=20000
)

print("Vocabulary size:", len(vocab))
print("First 20:", list(vocab.items())[:20])

Vocabulary size: 2450
First 20: [('<pad>', 0), ('<unk>', 1), ('.', 2), ('to', 3), ('the', 4), (',', 5), ('you', 6), ('and', 7), ('a', 8), ('$NUM', 9), ('i', 10), ('of', 11), ('for', 12), ('in', 13), ('please', 14), ('your', 15), ('if', 16), ('on', 17), ('we', 18), ('that', 19)]


In [23]:
from pathlib import Path
import numpy as np
import pandas as pd
from gensim.models import KeyedVectors

# Paths
google_path = "/kaggle/working/GoogleNews-vectors-negative300.bin.gz"
output_path = Path("/kaggle/working/mailsense/embeddings/word2vec.bin")
output_path.parent.mkdir(parents=True, exist_ok=True)

# Load the vocabulary we already built
train = pd.read_csv("/kaggle/working/mailsense/data/splits/train.csv")

from src.lstm.train_lstm import build_vocab

vocab = build_vocab(
    train["text"].tolist(),
    min_freq=2,
    max_size=20000,
)

print("MailSense vocabulary:", len(vocab))

# ---------------------------------------------------------
# Load pretrained Google News vectors
# ---------------------------------------------------------
print("\nLoading Google News Word2Vec...")
google_vectors = KeyedVectors.load_word2vec_format(
    google_path,
    binary=True,
)

print("Google News vocabulary:", len(google_vectors))
print("Embedding dimension:", google_vectors.vector_size)

# ---------------------------------------------------------
# Extract only words needed by MailSense
# ---------------------------------------------------------
words = []
vectors = []

for word, idx in vocab.items():
    # Do not include special tokens
    if word in {"<pad>", "<unk>"}:
        continue

    if word in google_vectors:
        words.append(word)
        vectors.append(google_vectors[word])

vectors = np.asarray(vectors, dtype=np.float32)

print("\nMatched words:", len(words))
print("Missing words:", len(vocab) - 2 - len(words))

coverage = len(words) / max(1, len(vocab) - 2)
print(f"Coverage: {coverage:.2%}")

# ---------------------------------------------------------
# Create compact KeyedVectors file
# ---------------------------------------------------------
compact_vectors = KeyedVectors(
    vector_size=google_vectors.vector_size
)

compact_vectors.add_vectors(words, vectors)

compact_vectors.save_word2vec_format(
    str(output_path),
    binary=True,
)

print("\nSaved:", output_path)
print("Size (MB):", round(output_path.stat().st_size / 1024**2, 2))

MailSense vocabulary: 2450

Loading Google News Word2Vec...


Google News vocabulary: 3000000
Embedding dimension: 300

Matched words: 2338
Missing words: 110
Coverage: 95.51%

Saved: /kaggle/working/mailsense/embeddings/word2vec.bin
Size (MB): 2.69


In [24]:
from pathlib import Path
from gensim.models import KeyedVectors

path = Path("/kaggle/working/mailsense/embeddings/word2vec.bin")

print("Exists:", path.exists())
print("Size (MB):", round(path.stat().st_size / 1024**2, 2))

vectors = KeyedVectors.load_word2vec_format(
    str(path),
    binary=True,
)

print("Vocabulary:", len(vectors))
print("Dimension:", vectors.vector_size)
print("Sample words:", vectors.index_to_key[:10])

Exists: True
Size (MB): 2.69
Vocabulary: 2338
Dimension: 300
Sample words: ['the', 'you', 'i', 'for', 'in', 'please', 'your', 'if', 'on', 'we']


In [25]:
%cd /kaggle/working/mailsense

/kaggle/working/mailsense


In [26]:
!python -m src.tfidf.train_tfidf

Train: 2555 | Validation: 548 | Test: 548
Vocabulary size: 6572

=== Model A: TF-IDF + Logistic Regression | val ===
Accuracy : 0.7974
Precision: 0.7976   (positive class = Actionable/Yes)
Recall   : 0.7636
F1-score : 0.7802
ROC-AUC  : 0.8787
Confusion matrix (rows=true [No,Yes], cols=pred [No,Yes]): [[240, 50], [61, 197]]
TP=197  TN=240  FP=50  FN=61
                     precision    recall  f1-score   support

Non-Actionable (No)       0.80      0.83      0.81       290
   Actionable (Yes)       0.80      0.76      0.78       258

           accuracy                           0.80       548
          macro avg       0.80      0.80      0.80       548
       weighted avg       0.80      0.80      0.80       548


=== Model A: TF-IDF + Logistic Regression | test ===
Accuracy : 0.7774
Precision: 0.7698   (positive class = Actionable/Yes)
Recall   : 0.7519
F1-score : 0.7608
ROC-AUC  : 0.8607
Confusion matrix (rows=true [No,Yes], cols=pred [No,Yes]): [[232, 58], [64, 194]]
TP=194  TN=232 

In [27]:
!python -m src.lstm.train_lstm --config configs/lstm.json

Device: cuda
train=2555 val=548 test=548 vocab=2450
Loading pretrained embeddings: embeddings/word2vec.bin
Embedding vocabulary: 2338
Embedding dimension: 300
Vocabulary coverage: 95.51%
Trainable parameters: 1,175,834
epoch 01 train_loss=0.5618 val_loss=0.4428 val_acc=0.8120 val_f1=0.7841
epoch 02 train_loss=0.3761 val_loss=0.4507 val_acc=0.8175 val_f1=0.7925
epoch 03 train_loss=0.2783 val_loss=0.4073 val_acc=0.8248 val_f1=0.8175
epoch 04 train_loss=0.1981 val_loss=0.4670 val_acc=0.8248 val_f1=0.8041
epoch 05 train_loss=0.1455 val_loss=0.5336 val_acc=0.8139 val_f1=0.8016
epoch 06 train_loss=0.1023 val_loss=0.5662 val_acc=0.8084 val_f1=0.8015
epoch 07 train_loss=0.0782 val_loss=0.6504 val_acc=0.8248 val_f1=0.8147
Early stopping at epoch 7; best epoch 3.

=== Model B: LSTM | val ===
Accuracy : 0.8248
Precision: 0.8022   (positive class = Actionable/Yes)
Recall   : 0.8333
F1-score : 0.8175
ROC-AUC  : 0.8996
Confusion matrix (rows=true [No,Yes], cols=pred [No,Yes]): [[237, 53], [43, 215]]

In [28]:
%cd /kaggle/working/mailsense
!python -m src.bert.train_bert --config configs/bert.json

/kaggle/working/mailsense
Device: cuda
Pretrained model: bert-base-uncased
Train: 2555 | Validation: 548 | Test: 548
config.json: 100%|█████████████████████████████| 570/570 [00:00<00:00, 2.07MB/s]
tokenizer_config.json: 100%|██████████████████| 48.0/48.0 [00:00<00:00, 355kB/s]
vocab.txt: 232kB [00:00, 18.6MB/s]
tokenizer.json: 466kB [00:00, 33.1MB/s]
model.safetensors: 100%|██████████████████████| 440M/440M [00:02<00:00, 167MB/s]
Loading weights: 100%|█| 199/199 [00:00<00:00, 1351.88it/s, Materializing param=
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.p

In [29]:
%cd /kaggle/working/mailsense
!python -m src.evaluation.compare

/kaggle/working/mailsense
# MailSense -- model comparison (test set)

Positive class = **Actionable (Yes)**. FN = an actionable e-mail predicted non-actionable (the costly error for this application).

| model | accuracy | precision | recall | f1 | roc_auc | TP | TN | FP | FN |
|---|---|---|---|---|---|---|---|---|---|
| Model A: TF-IDF + Logistic Regression | 0.7774 | 0.7698 | 0.7519 | 0.7608 | 0.8607 | 194 | 232 | 58 | 64 |
| Model B: LSTM | 0.7482 | 0.7174 | 0.7674 | 0.7416 | 0.8654 | 198 | 212 | 78 | 60 |
| Model C: BERT | 0.8485 | 0.8205 | 0.8682 | 0.8437 | 0.9316 | 224 | 241 | 49 | 34 |

Figures written to /kaggle/working/mailsense/results/figures
Wrote /kaggle/working/mailsense/results/comparison_test.csv


In [30]:
import pandas as pd

comparison = pd.read_csv("results/comparison_test.csv")
display(comparison)

,model,accuracy,precision,recall,f1,roc_auc,TP,TN,FP,FN
0,Model A: TF-IDF + Logistic Regression,0.777372,0.769841,0.751938,0.760784,0.860666,194,232,58,64
1,Model B: LSTM,0.748175,0.717391,0.767442,0.741573,0.865397,198,212,78,60
2,Model C: BERT,0.848540,0.820513,0.868217,0.843691,0.931649,224,241,49,34


In [31]:
%cd /kaggle/working/mailsense

!find results -maxdepth 2 -type f | sort

/kaggle/working/mailsense
results/bert/confusion_matrix_test.csv
results/bert/confusion_matrix_val.csv
results/bert/metrics.json
results/bert/test_predictions.csv
results/bert/test_summary.csv
results/bert/training_history.csv
results/comparison_test.csv
results/comparison_test.md
results/data_quality_report.json
results/figures/cm_bert_test.png
results/figures/cm_lstm_test.png
results/figures/cm_tfidf_logreg_test.png
results/lstm/confusion_matrix_test.csv
results/lstm/confusion_matrix_val.csv
results/lstm/metrics.json
results/lstm/test_predictions.csv
results/lstm/test_summary.csv
results/lstm/training_history.csv
results/removed_examples.csv
results/tfidf_logreg/confusion_matrix_test.csv
results/tfidf_logreg/confusion_matrix_val.csv
results/tfidf_logreg/metrics.json
results/tfidf_logreg/test_predictions.csv
results/tfidf_logreg/test_summary.csv


In [32]:
import json
from pathlib import Path

for model in ["tfidf_logreg", "lstm", "bert"]:
    path = Path(f"results/{model}/metrics.json")

    print("\n" + "="*60)
    print(model.upper())
    print("="*60)

    with open(path) as f:
        data = json.load(f)

    print(json.dumps(data, indent=2))


TFIDF_LOGREG
{
  "model": "tfidf_logreg",
  "config": {
    "model_name": "tfidf_logreg",
    "seed": 42,
    "splits_dir": "/kaggle/working/mailsense/data/splits",
    "results_dir": "/kaggle/working/mailsense/results",
    "models_dir": "/kaggle/working/mailsense/models",
    "lowercase": true,
    "ngram_range": [
      1,
      2
    ],
    "min_df": 2,
    "max_df": 0.9,
    "sublinear_tf": true,
    "max_features": null,
    "C": 1.0,
    "class_weight": "balanced",
    "max_iter": 2000
  },
  "metrics": {
    "val": {
      "accuracy": 0.7974452554744526,
      "precision": 0.7975708502024291,
      "recall": 0.7635658914728682,
      "f1": 0.7801980198019802,
      "macro_f1": 0.7961903804593657,
      "weighted_f1": 0.7971242409357094,
      "true_positives": 197,
      "true_negatives": 240,
      "false_positives": 50,
      "false_negatives": 61,
      "confusion_matrix": [
        [
          240,
          50
        ],
        [
          61,
          197
        ]
   

In [33]:
import pandas as pd

comparison = pd.read_csv("results/comparison_test.csv")

display(comparison)

comparison.to_csv(
    "results/final_test_comparison.csv",
    index=False
)

print("Saved: results/final_test_comparison.csv")

,model,accuracy,precision,recall,f1,roc_auc,TP,TN,FP,FN
0,Model A: TF-IDF + Logistic Regression,0.777372,0.769841,0.751938,0.760784,0.860666,194,232,58,64
1,Model B: LSTM,0.748175,0.717391,0.767442,0.741573,0.865397,198,212,78,60
2,Model C: BERT,0.848540,0.820513,0.868217,0.843691,0.931649,224,241,49,34


Saved: results/final_test_comparison.csv


In [34]:
from pathlib import Path

for model in ["tfidf_logreg", "lstm", "bert"]:
    print("\n", "="*60)
    print(model)

    for p in sorted(Path(f"results/{model}").glob("*")):
        print(p)


tfidf_logreg
results/tfidf_logreg/confusion_matrix_test.csv
results/tfidf_logreg/confusion_matrix_val.csv
results/tfidf_logreg/metrics.json
results/tfidf_logreg/test_predictions.csv
results/tfidf_logreg/test_summary.csv

lstm
results/lstm/confusion_matrix_test.csv
results/lstm/confusion_matrix_val.csv
results/lstm/metrics.json
results/lstm/test_predictions.csv
results/lstm/test_summary.csv
results/lstm/training_history.csv

bert
results/bert/confusion_matrix_test.csv
results/bert/confusion_matrix_val.csv
results/bert/metrics.json
results/bert/test_predictions.csv
results/bert/test_summary.csv
results/bert/training_history.csv


In [37]:
%cd /kaggle/working/mailsense

/kaggle/working/mailsense


In [39]:
import pandas as pd

history = pd.read_csv("results/lstm/training_history.csv")
display(history)

,epoch,train_loss,val_loss,val_accuracy,val_f1,seconds
0,1,0.561764,0.442837,0.812044,0.784067,1.9
1,2,0.376073,0.450679,0.817518,0.792531,0.9
2,3,0.278295,0.407296,0.824818,0.817490,0.8
3,4,0.198086,0.467007,0.824818,0.804082,0.8
4,5,0.145483,0.533615,0.813869,0.801556,0.8
5,6,0.102330,0.566196,0.808394,0.801512,0.9
6,7,0.078177,0.650370,0.824818,0.814672,0.8


In [41]:
import pandas as pd

history = pd.read_csv("results/bert/training_history.csv")
display(history)

,epoch,train_loss,val_loss,val_accuracy,val_f1,seconds
0,1,0.517253,0.355036,0.832117,0.836879,30.4
1,2,0.271376,0.367761,0.857664,0.839506,31.0
2,3,0.153206,0.458018,0.866788,0.860421,32.5
3,4,0.077515,0.539548,0.877737,0.873823,34.4


In [43]:
import shutil
from pathlib import Path

repo = Path("/kaggle/working/mailsense")
backup = Path("/kaggle/working/MailSense_Final_Backup")

if backup.exists():
    shutil.rmtree(backup)

backup.mkdir()

# Final datasets
for src in [
    repo / "data/processed/mailsense_clean.csv",
    repo / "data/splits/train.csv",
    repo / "data/splits/val.csv",
    repo / "data/splits/test.csv",
]:
    if src.exists():
        dst = backup / "data" / src.relative_to(repo / "data")
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

# Results
if (repo / "results").exists():
    shutil.copytree(repo / "results", backup / "results")

# Models
if (repo / "models").exists():
    shutil.copytree(repo / "models", backup / "models")

# Compact pretrained embeddings
if (repo / "embeddings").exists():
    shutil.copytree(repo / "embeddings", backup / "embeddings")

# Configs
if (repo / "configs").exists():
    shutil.copytree(repo / "configs", backup / "configs")

# Create archive
archive = shutil.make_archive(
    "/kaggle/working/MailSense_Final_Backup",
    "zip",
    backup
)

print("Created:")
print(archive)

print("Size (MB):", round(Path(archive).stat().st_size / 1024**2, 2))

Created:
/kaggle/working/MailSense_Final_Backup.zip
Size (MB): 784.03


In [44]:
from pathlib import Path

backup = Path("/kaggle/working/MailSense_Final_Backup")

for path in sorted(backup.rglob("*")):
    if path.is_file():
        print(path.relative_to(backup), 
              f"({path.stat().st_size / 1024**2:.2f} MB)")

configs/bert.json (0.00 MB)
configs/lstm.json (0.00 MB)
configs/tfidf.json (0.00 MB)
data/processed/mailsense_clean.csv (0.70 MB)
data/splits/test.csv (0.11 MB)
data/splits/train.csv (0.49 MB)
data/splits/val.csv (0.11 MB)
embeddings/word2vec.bin (2.69 MB)
models/bert/best.pt (417.71 MB)
models/bert/best_hf/config.json (0.00 MB)
models/bert/best_hf/model.safetensors (417.67 MB)
models/bert/best_hf/tokenizer.json (0.68 MB)
models/bert/best_hf/tokenizer_config.json (0.00 MB)
models/bert/config.json (0.00 MB)
models/lstm/best.pt (4.49 MB)
models/lstm/config.json (0.00 MB)
models/lstm/final.pt (4.49 MB)
models/lstm/vocab.json (0.04 MB)
models/tfidf_logreg/config.json (0.00 MB)
models/tfidf_logreg/pipeline.joblib (0.19 MB)
results/bert/confusion_matrix_test.csv (0.00 MB)
results/bert/confusion_matrix_val.csv (0.00 MB)
results/bert/metrics.json (0.00 MB)
results/bert/test_predictions.csv (0.07 MB)
results/bert/test_summary.csv (0.00 MB)
results/bert/training_history.csv (0.00 MB)
results/com

In [45]:
import shutil
from pathlib import Path

repo = Path("/kaggle/working/mailsense")
backup = Path("/kaggle/working/MailSense_Final_Backup")

src = repo / "data/Ask0729-fixed.txt"
dst = backup / "data/Ask0729-fixed.txt"

if src.exists():
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print("Raw dataset copied.")
else:
    print("Raw dataset not found!")

Raw dataset copied.


In [46]:
from pathlib import Path

backup = Path("/kaggle/working/MailSense_Final_Backup")

print("=" * 80)
print("CONTENTS OF MailSense_Final_Backup")
print("=" * 80)

files = sorted([p for p in backup.rglob("*") if p.is_file()])

total_size = 0

for p in files:
    size_mb = p.stat().st_size / (1024 ** 2)
    total_size += size_mb
    print(f"{p.relative_to(backup)}    [{size_mb:.2f} MB]")

print("\n" + "=" * 80)
print(f"Total files: {len(files)}")
print(f"Total size: {total_size:.2f} MB")
print("=" * 80)

CONTENTS OF MailSense_Final_Backup
configs/bert.json    [0.00 MB]
configs/lstm.json    [0.00 MB]
configs/tfidf.json    [0.00 MB]
data/Ask0729-fixed.txt    [0.33 MB]
data/processed/mailsense_clean.csv    [0.70 MB]
data/splits/test.csv    [0.11 MB]
data/splits/train.csv    [0.49 MB]
data/splits/val.csv    [0.11 MB]
embeddings/word2vec.bin    [2.69 MB]
models/bert/best.pt    [417.71 MB]
models/bert/best_hf/config.json    [0.00 MB]
models/bert/best_hf/model.safetensors    [417.67 MB]
models/bert/best_hf/tokenizer.json    [0.68 MB]
models/bert/best_hf/tokenizer_config.json    [0.00 MB]
models/bert/config.json    [0.00 MB]
models/lstm/best.pt    [4.49 MB]
models/lstm/config.json    [0.00 MB]
models/lstm/final.pt    [4.49 MB]
models/lstm/vocab.json    [0.04 MB]
models/tfidf_logreg/config.json    [0.00 MB]
models/tfidf_logreg/pipeline.joblib    [0.19 MB]
results/bert/confusion_matrix_test.csv    [0.00 MB]
results/bert/confusion_matrix_val.csv    [0.00 MB]
results/bert/metrics.json    [0.00 MB]

In [47]:
from pathlib import Path

duplicate = Path("/kaggle/working/MailSense_Final_Backup/models/bert/best.pt")

if duplicate.exists():
    duplicate.unlink()
    print("Removed duplicate BERT best.pt")
else:
    print("best.pt already absent")

Removed duplicate BERT best.pt


In [48]:
from pathlib import Path

backup = Path("/kaggle/working/MailSense_Final_Backup")

print("=" * 80)
print("FINAL BACKUP CONTENTS")
print("=" * 80)

files = sorted(p for p in backup.rglob("*") if p.is_file())

total_size = 0

for p in files:
    size_mb = p.stat().st_size / (1024 ** 2)
    total_size += size_mb
    print(f"{p.relative_to(backup)}    [{size_mb:.2f} MB]")

print("\n" + "=" * 80)
print(f"Total files: {len(files)}")
print(f"Total size: {total_size:.2f} MB")
print("=" * 80)

FINAL BACKUP CONTENTS
configs/bert.json    [0.00 MB]
configs/lstm.json    [0.00 MB]
configs/tfidf.json    [0.00 MB]
data/Ask0729-fixed.txt    [0.33 MB]
data/processed/mailsense_clean.csv    [0.70 MB]
data/splits/test.csv    [0.11 MB]
data/splits/train.csv    [0.49 MB]
data/splits/val.csv    [0.11 MB]
embeddings/word2vec.bin    [2.69 MB]
models/bert/best_hf/config.json    [0.00 MB]
models/bert/best_hf/model.safetensors    [417.67 MB]
models/bert/best_hf/tokenizer.json    [0.68 MB]
models/bert/best_hf/tokenizer_config.json    [0.00 MB]
models/bert/config.json    [0.00 MB]
models/lstm/best.pt    [4.49 MB]
models/lstm/config.json    [0.00 MB]
models/lstm/final.pt    [4.49 MB]
models/lstm/vocab.json    [0.04 MB]
models/tfidf_logreg/config.json    [0.00 MB]
models/tfidf_logreg/pipeline.joblib    [0.19 MB]
results/bert/confusion_matrix_test.csv    [0.00 MB]
results/bert/confusion_matrix_val.csv    [0.00 MB]
results/bert/metrics.json    [0.00 MB]
results/bert/test_predictions.csv    [0.07 MB]


In [49]:
import shutil
from pathlib import Path

backup = Path("/kaggle/working/MailSense_Final_Backup")

zip_path = shutil.make_archive(
    "/kaggle/working/MailSense_Final_Backup",
    "zip",
    backup
)

print("Created:", zip_path)
print("Size:", round(Path(zip_path).stat().st_size / 1024**2, 2), "MB")

Created: /kaggle/working/MailSense_Final_Backup.zip
Size: 397.34 MB


In [50]:
%cd /kaggle/working/mailsense

import json
import re
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from gensim.models import KeyedVectors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from src.preprocessing.clean import clean_text
from src.lstm.model import LSTMClassifier


# ============================================================
# 1. Test emails
# ============================================================

emails = [
    "Please submit the completed project report by 5 PM tomorrow.",

    "The department meeting was held yesterday. The discussion went well.",

    "Could you review the attached document and send me your feedback?",

    "Thank you for attending the seminar. We hope you enjoyed the event.",

    "Your appointment has been confirmed for Monday at 10 AM. Please arrive 15 minutes early."
]


# ============================================================
# 2. Model A — TF-IDF + Logistic Regression
# ============================================================

print("Loading TF-IDF model...")

tfidf_model = torch.load(
    "models/tfidf_logreg/pipeline.joblib",
    map_location="cpu"
) if False else None

# joblib is the correct loader
import joblib
tfidf_model = joblib.load(
    "models/tfidf_logreg/pipeline.joblib"
)


# ============================================================
# 3. Model B — LSTM + Word2Vec
# ============================================================

print("Loading LSTM model...")

with open("models/lstm/vocab.json", "r") as f:
    vocab = json.load(f)

word2vec = KeyedVectors.load_word2vec_format(
    "embeddings/word2vec.bin",
    binary=True
)

checkpoint = torch.load(
    "models/lstm/best.pt",
    map_location="cpu"
)

# Build embedding matrix
embedding_dim = word2vec.vector_size
embedding_matrix = np.zeros(
    (len(vocab), embedding_dim),
    dtype=np.float32
)

for word, idx in vocab.items():
    if word in ("<pad>", "<unk>"):
        continue

    if word in word2vec:
        embedding_matrix[idx] = word2vec[word]

embedding_matrix = torch.tensor(embedding_matrix)

lstm_model = LSTMClassifier(
    embedding_matrix=embedding_matrix,
    hidden_dim=128,
    num_layers=1,
    bidirectional=True,
    dropout=0.3,
    num_classes=2,
    pad_id=vocab["<pad>"],
    embedding_trainable=True,
)

# Handle either a raw state_dict or checkpoint dictionary
if "model_state_dict" in checkpoint:
    lstm_model.load_state_dict(checkpoint["model_state_dict"])
elif "state_dict" in checkpoint:
    lstm_model.load_state_dict(checkpoint["state_dict"])
else:
    lstm_model.load_state_dict(checkpoint)

lstm_model.eval()


def lstm_encode(text, max_len=64):
    tokens = clean_text(text).lower().split()

    ids = [
        vocab.get(token, vocab["<unk>"])
        for token in tokens
    ]

    ids = ids[:max_len]

    if len(ids) < max_len:
        ids += [vocab["<pad>"]] * (max_len - len(ids))

    return torch.tensor([ids], dtype=torch.long)


# ============================================================
# 4. Model C — BERT
# ============================================================

print("Loading BERT model...")

bert_path = "models/bert/best_hf"

bert_tokenizer = AutoTokenizer.from_pretrained(
    bert_path,
    local_files_only=True
)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    bert_path,
    local_files_only=True
)

bert_model.eval()


# ============================================================
# 5. Run predictions
# ============================================================

print("\n" + "=" * 90)
print("MailSense Inference Demo")
print("=" * 90)

for i, email in enumerate(emails, 1):

    print(f"\nEMAIL {i}")
    print("-" * 90)
    print(email)

    cleaned = clean_text(email)

    # --------------------------------------------------------
    # TF-IDF
    # --------------------------------------------------------
    tfidf_prob = tfidf_model.predict_proba([cleaned])[0]
    tfidf_pred = int(np.argmax(tfidf_prob))

    # --------------------------------------------------------
    # LSTM
    # --------------------------------------------------------
    input_ids = lstm_encode(cleaned)

    with torch.no_grad():
        logits = lstm_model(input_ids)
        probs = torch.softmax(logits, dim=1)[0]
        lstm_pred = int(torch.argmax(probs))

    lstm_conf = float(probs[lstm_pred])

    # --------------------------------------------------------
    # BERT
    # --------------------------------------------------------
    inputs = bert_tokenizer(
        cleaned,
        return_tensors="pt",
        truncation=True,
        max_length=64
    )

    with torch.no_grad():
        outputs = bert_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)[0]
        bert_pred = int(torch.argmax(probs))

    bert_conf = float(probs[bert_pred])

    labels = {
        0: "Non-Actionable",
        1: "Actionable"
    }

    print("\nPredictions:")
    print(
        f"TF-IDF + LR : {labels[tfidf_pred]}"
        f"  ({tfidf_prob[tfidf_pred] * 100:.2f}%)"
    )

    print(
        f"LSTM        : {labels[lstm_pred]}"
        f"  ({lstm_conf * 100:.2f}%)"
    )

    print(
        f"BERT        : {labels[bert_pred]}"
        f"  ({bert_conf * 100:.2f}%)"
    )

print("\n" + "=" * 90)
print("Inference completed.")
print("=" * 90)

/kaggle/working/mailsense
Loading TF-IDF model...
Loading LSTM model...
Loading BERT model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


MailSense Inference Demo

EMAIL 1
------------------------------------------------------------------------------------------
Please submit the completed project report by 5 PM tomorrow.

Predictions:
TF-IDF + LR : Actionable  (62.51%)
LSTM        : Actionable  (99.20%)
BERT        : Actionable  (99.78%)

EMAIL 2
------------------------------------------------------------------------------------------
The department meeting was held yesterday. The discussion went well.

Predictions:
TF-IDF + LR : Non-Actionable  (68.77%)
LSTM        : Non-Actionable  (95.19%)
BERT        : Non-Actionable  (99.72%)

EMAIL 3
------------------------------------------------------------------------------------------
Could you review the attached document and send me your feedback?

Predictions:
TF-IDF + LR : Actionable  (86.41%)
LSTM        : Actionable  (95.91%)
BERT        : Actionable  (99.82%)

EMAIL 4
------------------------------------------------------------------------------------------
Thank you